# InsightFace Example

## InsightFace — Face Detection & Analysis

This notebook is an example of face detection and analysis with **InsightFace**.

```
Image (URL or file) → InsightFace → Annotated image + bounding boxes + landmarks + age/gender
```

InsightFace detects all faces in an image and for each face provides:
- **Bounding box** — face location
- **Landmarks** — 5 keypoints (eyes, nose, mouth corners)
- **Age & Gender** — estimated age and gender

### Available models

| Model | Description | Speed |
|-------|-------------|-------|
| `buffalo_l` | Largest, most accurate (det + align + recog + 3D + age/gender) | slowest |
| `buffalo_m` | Medium, age/gender included | — |
| `buffalo_s` | Small, fastest, no age/gender | fastest |
| `buffalo_sc` | Tiny model, for CCTV / low-res | — |

Change `MODEL_NAME` in the writefile cell to switch between them.

📄 [InsightFace GitHub](https://github.com/deepinsight/insightface)

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

PROJECT_PATH = "/content/drive/MyDrive/CV-Samples/face"
!mkdir -p "{PROJECT_PATH}"
%cd "{PROJECT_PATH}"
!ls


In [ ]:
# Download sample images from GitHub
import os

BASE_URL = "https://raw.githubusercontent.com/mastnk/cv-samples/main/face"
IMAGE_FILES = [
    "sunriseforever-family-7638971_640.jpg",
]
for fname in IMAGE_FILES:
    url  = f'{BASE_URL}/{fname}'
    dest = f'{PROJECT_PATH}/{fname}'
    if not os.path.exists(dest):
        os.system(f'wget -q "{url}" -O "{dest}"')
        print(f'Downloaded: {fname}')
    else:
        print(f'Already exists: {fname}')

%cd "{PROJECT_PATH}"
!ls


In [ ]:
# Install InsightFace
!pip install insightface onnxruntime -q


## Using Your Own Images

There are two ways to provide images for face analysis:

**① Specify a URL**  
Pass a direct image URL to the `--url` flag when running the script:
```
%run insightface.py --url https://cdn.pixabay.com/photo/2021/04/17/18/16/asian-6186466_640.jpg
```

**② Upload images to the `PROJECT_PATH/` folder**  
Place your image files in `PROJECT_PATH/`, then run the script with `--file` or `--dir`.

The easiest way to upload is through **Google Drive**:  
Open [drive.google.com](https://drive.google.com), navigate to `My Drive / CV-Samples / face/`, and drag & drop your files there.  
They will be immediately available in Colab via the mounted drive — no extra upload step needed.

```
%run insightface.py --file your_image.jpg   # single file
%run insightface.py --dir  .                # all images in folder
```

## Selecting a Model

In the writefile cell below, change `MODEL_NAME` to switch models.  
When multiple `MODEL_NAME = ...` lines are listed, **only the last one takes effect**.

```python
# MODEL_NAME = 'buffalo_l'   # most accurate (det + align + recog + 3D + age/gender)
# MODEL_NAME = 'buffalo_m'   # medium, age/gender included
# MODEL_NAME = 'buffalo_sc'  # tiny model, for CCTV / low-res
MODEL_NAME   = 'buffalo_s'   # fastest, no age/gender  ← active
```

Larger models are more accurate but take longer to run. Start with `buffalo_s` for quick experiments.  
Use `buffalo_l` or `buffalo_m` if you need age and gender estimation.

In [ ]:
# Save this cell as a Python file (Execute after editing)
%%writefile face.py
"""InsightFace Face Detection & Analysis — command-line interface.

Usage:
  %run insightface.py --url  <image_url>  [--disp]
  %run insightface.py --file <image_path> [--disp]
  %run insightface.py --dir  <image_dir>  [--disp]
"""

import argparse
import glob
import os

import cv2
import numpy as np
from PIL import Image
import requests
from io import BytesIO
from insightface.app import FaceAnalysis

# ---- IPython display (works when executed via %run in Colab) -----
try:
    from IPython.display import display as ipy_display
    _has_ipy = True
except ImportError:
    _has_ipy = False

# ---- Configuration -----------------------------------------------
MODEL_NAME   = 'buffalo_s'   # fastest, no age/gender
# MODEL_NAME = 'buffalo_m'   # medium, age/gender included
# MODEL_NAME = 'buffalo_l'   # most accurate (det + align + recog + 3D + age/gender)
# MODEL_NAME = 'buffalo_sc'  # tiny model, for CCTV / low-res

PROJECT_PATH = '/content/drive/MyDrive/CV-Samples/face'

# ---- Model loading -----------------------------------------------
app = FaceAnalysis(
    name=MODEL_NAME,
    providers=['CUDAExecutionProvider', 'CPUExecutionProvider'],
)
app.prepare(ctx_id=0, det_size=(640, 640))

# ---- Display helper ----------------------------------------------
def show(annotated: np.ndarray, label: str, disp: bool) -> None:
    """When --disp is active, display the annotated image then print the filename/URL."""
    if disp:
        if _has_ipy:
            ipy_display(Image.fromarray(annotated[:, :, ::-1]))  # BGR -> RGB
        print(label)

# ---- Helper: print face info ------------------------------------
def print_face_info(faces):
    """Print detection score, age, and gender for each detected face."""
    print(f'  Faces detected: {len(faces)}')
    for i, face in enumerate(faces):
        info = f'  Face {i+1}: det_score={face.det_score:.2f}'
        if hasattr(face, 'age') and face.age is not None:
            info += f'  age={int(face.age)}'
        if hasattr(face, 'gender') and face.gender is not None:
            gender_str = 'M' if face.gender == 1 else 'F'
            info += f'  gender={gender_str}'
        print(info)

# ---- Functions ---------------------------------------------------
def analyze_url(url: str, disp: bool = False):
    """Download an image from a URL and analyze faces."""
    image = np.array(Image.open(BytesIO(requests.get(url).content)).convert('RGB'))[:, :, ::-1]  # RGB -> BGR
    faces = app.get(image)
    annotated = app.draw_on(image, faces)
    show(annotated, url, disp)
    print_face_info(faces)
    return faces


def analyze_file(path: str, disp: bool = False):
    """Analyze faces in a single local image file."""
    image = cv2.imread(path)
    faces = app.get(image)
    annotated = app.draw_on(image, faces)
    show(annotated, path, disp)
    print_face_info(faces)
    return faces


def analyze_dir(directory: str, disp: bool = False):
    """Analyze faces in all images in a directory."""
    exts = ['jpg', 'jpeg', 'png', 'bmp', 'JPG', 'JPEG', 'PNG', 'BMP']
    filepaths = []
    for ext in exts:
        filepaths += sorted(glob.glob(os.path.join(directory, f'*.{ext}')))

    if not filepaths:
        print(f'No images found in: {directory}')
        return

    for path in filepaths:
        analyze_file(path, disp)
        print()


# ---- Argument parsing --------------------------------------------
parser = argparse.ArgumentParser(description='InsightFace Face Detection & Analysis')
group  = parser.add_mutually_exclusive_group(required=True)
group.add_argument('--url',  type=str, help='Image URL to analyze')
group.add_argument('--file', type=str, help='Path to a single image file')
group.add_argument('--dir',  type=str, help='Directory of images to analyze')
parser.add_argument('--disp', action='store_true',
                    help='Display annotated image and filename/URL before results')
args = parser.parse_args()

# ---- Run ---------------------------------------------------------
if args.url:
    analyze_url(args.url,   disp=args.disp)
elif args.file:
    analyze_file(args.file, disp=args.disp)
elif args.dir:
    analyze_dir(args.dir,   disp=args.disp)


## `face.py` Usage

After running the `%%writefile` cell above, `face.py` is saved in `PROJECT_PATH`.
Run it with **`%run`** (not `!python`) so that inline image display works in Colab.

```
%run face.py --url  <image_url>        # analyze faces in a remote image
%run face.py --file <image_path>       # analyze faces in a single local file
%run face.py --dir  <image_directory>  # analyze faces in all images in a folder
```

**Optional arguments**

| Flag | Default | Description |
|------|---------|-------------|
| `--disp` | off | Display annotated image and filename / URL before results |

**Examples**

```bash
# Analyze faces in a remote image and display it
%run face.py --url https://cdn.pixabay.com/photo/2021/04/17/18/16/asian-6186466_640.jpg --disp

# Analyze faces in one file
%run face.py --file sunriseforever-family-7638971_640.jpg --disp

# Analyze faces in every image in the folder
%run face.py --dir . --disp
```

**Output format (with `--disp`)**

```
<annotated image with bounding boxes and landmarks displayed inline>
sunriseforever-family-7638971_640.jpg
  Faces detected: 3
  Face 1: det_score=0.98  age=34  gender=F
  Face 2: det_score=0.95  age=8   gender=M
  Face 3: det_score=0.91  age=40  gender=M
```

> **Note:** Age and gender are only available with `buffalo_m` or `buffalo_l` models.

## Execution Methods

Use **`%run`** to execute `face.py` inside the Colab kernel — this enables inline image display with `--disp`.

| Mode | Flag | Description |
|------|------|-------------|
| From URL | `--url <url>` | Fetches and analyzes faces in a remote image |
| Single file | `--file <path>` | Analyzes faces in one local image |
| Directory | `--dir <path>` | Analyzes faces in all images in a folder |

Add `--disp` to display each annotated image (with bounding boxes and landmarks) and its filename / URL before the results.

> **Tip:** Switch to `buffalo_l` or `buffalo_m` in the writefile cell to enable age and gender estimation.

In [ ]:
# Execute: analyze faces in an image from URL (with display)
%cd "{PROJECT_PATH}"
%run face.py --disp --url https://cdn.pixabay.com/photo/2021/04/17/18/16/asian-6186466_640.jpg


In [ ]:
# Execute: analyze faces in a single local image file (with display)
%cd "{PROJECT_PATH}"
%run face.py --disp --file sunriseforever-family-7638971_640.jpg


In [ ]:
# Execute: analyze faces in all images in a directory (with display)
%cd "{PROJECT_PATH}"
%run face.py --disp --dir .


💾 **Don't forget to save this notebook!**

To keep your work, go to **File → Save a copy in Drive** before closing.